# Bharatanatyam Mudra Gesture Recognition
## Training Pipeline using MediaPipe Model Maker

This notebook trains a MediaPipe gesture recognition model on Bharatanatyam hand mudras.

**Pipeline:**
1. Organize images into class-label subfolders
2. Augment data (50 variants per mudra) using `albumentations`
3. Load with MediaPipe Model Maker `Dataset.from_folder()`
4. Split into train/validation/test sets
5. Train and evaluate
6. Export `.task` model bundle
7. Run inference on a sample image

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
for folder in os.listdir("/content/drive/MyDrive/mudras"):
    print(folder)

pataka
tripataka
ardhapataka
kartarimukha
mayura
ardhachandra
arala
sukatunda
musti
sikhara
kapittha
katakamukha
suchi
chandrakala
padmakosa
sarpashirsa
mrgasirsa
simhamukha
kangula
alapadma
catura
bramara
hamsasya
hamsapaksa
sandamsa
mukula
tamracuda
trisula


In [3]:
!pip install --upgrade pip
!pip install mediapipe-model-maker

In [4]:
pip install -q mediapipe albumentations matplotlib seaborn

In [5]:
import os
import shutil
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import albumentations as A
from pathlib import Path

## 1. Organize Images into Subfolder Structure

MediaPipe Model Maker expects `dataset_dir/class_name/image.jpg`.
If your `mudras/` folder has flat files (e.g. `mudras/alapadma.jpg`),
this cell reorganizes them into `mudras/alapadma/alapadma.jpg`.

In [6]:
MUDRAS_DIR = "/content/drive/MyDrive/mudras"
AUGMENTED_DIR = "/content/drive/MyDrive/mudras_augmented"
MODEL_DIR = "bharatanatyam_mudra_model"
EXPORT_NAME = "bharatanatyam_mudras.task"

# Check if images are flat files or already in subfolders
entries = os.listdir(MUDRAS_DIR)
flat_files = [e for e in entries if os.path.isfile(os.path.join(MUDRAS_DIR, e)) and e.lower().endswith(('.jpg', '.jpeg', '.png'))]

if flat_files:
    print(f"Found {len(flat_files)} flat image files. Reorganizing into subfolders...")
    for f in flat_files:
        class_name = os.path.splitext(f)[0]
        class_dir = os.path.join(MUDRAS_DIR, class_name)
        os.makedirs(class_dir, exist_ok=True)
        src = os.path.join(MUDRAS_DIR, f)
        dst = os.path.join(class_dir, f)
        if not os.path.exists(dst):
            shutil.move(src, dst)
    print("Done reorganizing.")
else:
    print("Images are already in subfolders.")

# List detected mudra classes
classes = sorted([
    d for d in os.listdir(MUDRAS_DIR)
    if os.path.isdir(os.path.join(MUDRAS_DIR, d)) and not d.startswith('.')
])
print(f"\nDetected {len(classes)} mudra classes:")
for i, c in enumerate(classes, 1):
    n_images = len(glob.glob(os.path.join(MUDRAS_DIR, c, '*.[jJpP][pPnN][gG]*')))
    print(f"  {i:2d}. {c} ({n_images} image(s))")

Images are already in subfolders.

Detected 28 mudra classes:
   1. alapadma (1 image(s))
   2. arala (1 image(s))
   3. ardhachandra (1 image(s))
   4. ardhapataka (1 image(s))
   5. bramara (1 image(s))
   6. catura (1 image(s))
   7. chandrakala (1 image(s))
   8. hamsapaksa (1 image(s))
   9. hamsasya (1 image(s))
  10. kangula (1 image(s))
  11. kapittha (1 image(s))
  12. kartarimukha (1 image(s))
  13. katakamukha (1 image(s))
  14. mayura (1 image(s))
  15. mrgasirsa (1 image(s))
  16. mukula (1 image(s))
  17. musti (1 image(s))
  18. padmakosa (1 image(s))
  19. pataka (1 image(s))
  20. sandamsa (1 image(s))
  21. sarpashirsa (1 image(s))
  22. sikhara (1 image(s))
  23. simhamukha (1 image(s))
  24. suchi (1 image(s))
  25. sukatunda (1 image(s))
  26. tamracuda (1 image(s))
  27. tripataka (1 image(s))
  28. trisula (1 image(s))


## 2. Data Augmentation

Since we only have ~1 image per class, we generate at least 50 augmented variants
per mudra using `albumentations` (rotations, flips, brightness/contrast, zoom, etc.).

In [7]:
NUM_AUGMENTED = 50  # augmented images to generate per original image

augmentation_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=30, p=0.8, border_mode=cv2.BORDER_REFLECT_101),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.8),
    A.RandomScale(scale_limit=0.15, p=0.5),
    A.GaussNoise(var_limit=(5.0, 30.0), p=0.3),
    A.GaussianBlur(blur_limit=(1, 3), p=0.2),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.CLAHE(clip_limit=2.0, p=0.2),
])

# Clean and recreate augmented directory
if os.path.exists(AUGMENTED_DIR):
    shutil.rmtree(AUGMENTED_DIR)

total_generated = 0

for class_name in classes:
    src_dir = os.path.join(MUDRAS_DIR, class_name)
    dst_dir = os.path.join(AUGMENTED_DIR, class_name)
    os.makedirs(dst_dir, exist_ok=True)

    # Gather all source images in this class folder
    src_images = glob.glob(os.path.join(src_dir, '*.[jJpP][pPnN][gG]*'))

    for img_path in src_images:
        image = cv2.imread(img_path)
        if image is None:
            print(f"  Warning: could not read {img_path}, skipping.")
            continue
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        basename = os.path.splitext(os.path.basename(img_path))[0]

        # Copy the original image
        orig_dst = os.path.join(dst_dir, os.path.basename(img_path))
        cv2.imwrite(orig_dst, cv2.cvtColor(image, cv2.COLOR_RGB2BGR))

        # Generate augmented variants
        for i in range(NUM_AUGMENTED):
            augmented = augmentation_pipeline(image=image)['image']
            aug_filename = f"{basename}_aug_{i:03d}.jpg"
            aug_path = os.path.join(dst_dir, aug_filename)
            cv2.imwrite(aug_path, cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR))

        generated = NUM_AUGMENTED + 1  # augmented + original
        total_generated += generated

    n_total = len(os.listdir(dst_dir))
    print(f"  {class_name}: {n_total} images")

print(f"\nTotal images in augmented dataset: {total_generated}")

/tmp/ipython-input-7-27022890.py:8: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5.0, 30.0), p=0.3),
/usr/local/lib/python3.11/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


  alapadma: 51 images
  arala: 51 images
  ardhachandra: 51 images
  ardhapataka: 51 images
  bramara: 51 images
  catura: 51 images
  chandrakala: 51 images
  hamsapaksa: 51 images
  hamsasya: 51 images
  kangula: 51 images
  kapittha: 51 images
  kartarimukha: 51 images
  katakamukha: 51 images
  mayura: 51 images
  mrgasirsa: 51 images
  mukula: 51 images
  musti: 51 images
  padmakosa: 51 images
  pataka: 51 images
  sandamsa: 51 images
  sarpashirsa: 51 images
  sikhara: 51 images
  simhamukha: 51 images
  suchi: 51 images
  sukatunda: 51 images
  tamracuda: 51 images
  tripataka: 51 images
  trisula: 51 images

Total images in augmented dataset: 1428


## 3. Load Dataset with MediaPipe Model Maker

In [9]:
import os
none_dir = os.path.join(AUGMENTED_DIR, "none")
os.makedirs(none_dir, exist_ok=True)
print("✅ none folder created!")

✅ none folder created!


In [10]:
from mediapipe_model_maker import gesture_recognizer

data = gesture_recognizer.Dataset.from_folder(
    dirname=AUGMENTED_DIR,
    hparams=gesture_recognizer.HandDataPreprocessingParams()
)

print(f"Dataset loaded: {len(data)} samples")

Dataset loaded: 968 samples


## 4. Split into Train / Validation / Test Sets

In [11]:
train_data, rest_data = data.split(0.8)
validation_data, test_data = rest_data.split(0.5)

print(f"Train:      {len(train_data)} samples")
print(f"Validation: {len(validation_data)} samples")
print(f"Test:       {len(test_data)} samples")

Train:      774 samples
Validation: 97 samples
Test:       97 samples


## 5. Train the Gesture Recognizer

In [13]:
hparams = gesture_recognizer.HParams(
    export_dir=MODEL_DIR,
    epochs=30,
    batch_size=8,
    learning_rate=0.001,
)

options = gesture_recognizer.GestureRecognizerOptions(
    hparams=hparams
)

model = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options,
)

print("Training complete.")

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 hand_embedding (InputLayer  [(None, 128)]             0         
 )                                                               
                                                                 
 batch_normalization (Batch  (None, 128)               512       
 Normalization)                                                  
                                                                 
 re_lu (ReLU)                (None, 128)               0         
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 custom_gesture_recognizer_  (None, 29)                3741      
 out (Dense)                                                     
                                                             

## 6. Evaluate on Test Set

In [14]:
loss, accuracy = model.evaluate(test_data)
print(f"Test Loss:     {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

4/4 [==============================] - 1s 15ms/step - loss: 0.2488 - categorical_accuracy: 0.8866
Test Loss:     0.2488
Test Accuracy: 0.8866


## 7. Confusion Matrix on Test Set

In [15]:
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
from collections import Counter

# Export a temporary model for inference-based confusion matrix
model.export_model(EXPORT_NAME)
print(f"Model exported to: {os.path.join(MODEL_DIR, EXPORT_NAME)}")

# Set up recognizer for confusion matrix
model_path = os.path.join(MODEL_DIR, EXPORT_NAME)

base_options = mp_python.BaseOptions(model_asset_path=model_path)
recognizer_options = vision.GestureRecognizerOptions(base_options=base_options)
recognizer = vision.GestureRecognizer.create_from_options(recognizer_options)

# Run inference on all test images to build confusion matrix
# Gather test images from the augmented directory
true_labels = []
pred_labels = []
label_names = sorted(os.listdir(AUGMENTED_DIR))

for class_name in label_names:
    class_dir = os.path.join(AUGMENTED_DIR, class_name)
    if not os.path.isdir(class_dir):
        continue
    images = glob.glob(os.path.join(class_dir, '*.jpg'))[:5]  # sample up to 5 per class
    for img_path in images:
        mp_image = mp.Image.create_from_file(img_path)
        result = recognizer.recognize(mp_image)
        if result.gestures and len(result.gestures) > 0:
            predicted = result.gestures[0][0].category_name
        else:
            predicted = "unknown"
        true_labels.append(class_name)
        pred_labels.append(predicted)

recognizer.close()

# Build and plot confusion matrix
from sklearn.metrics import confusion_matrix

all_labels = sorted(set(true_labels + pred_labels))
cm = confusion_matrix(true_labels, pred_labels, labels=all_labels)

fig, ax = plt.subplots(figsize=(max(12, len(all_labels) * 0.6), max(10, len(all_labels) * 0.5)))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=all_labels, yticklabels=all_labels, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Bharatanatyam Mudra Recognition')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

Using existing files at /tmp/model_maker/gesture_recognizer/palm_detection_full.tflite
Using existing files at /tmp/model_maker/gesture_recognizer/hand_landmark_full.tflite
Model exported to: bharatanatyam_mudra_model/bharatanatyam_mudras.task


## 8. Export Final Model

In [16]:
export_path = os.path.join(MODEL_DIR, EXPORT_NAME)
print(f"Final model bundle: {export_path}")
print(f"File size: {os.path.getsize(export_path) / (1024*1024):.2f} MB")

Final model bundle: bharatanatyam_mudra_model/bharatanatyam_mudras.task
File size: 8.08 MB


## 9. Inference on a Sample Image

Pick one image from the original `mudras/` directory and run recognition.

In [17]:
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

# Find a sample image from the original mudras directory
sample_image_path = None
for class_name in classes:
    class_dir = os.path.join(MUDRAS_DIR, class_name)
    images = glob.glob(os.path.join(class_dir, '*.[jJpP][pPnN][gG]*'))
    if images:
        sample_image_path = images[0]
        sample_class = class_name
        break

if sample_image_path is None:
    print("No sample image found!")
else:
    print(f"Sample image: {sample_image_path}")
    print(f"True label:   {sample_class}")
    print()

    # Load and run inference
    model_path = os.path.join(MODEL_DIR, EXPORT_NAME)
    base_options = mp_python.BaseOptions(model_asset_path=model_path)
    recognizer_options = vision.GestureRecognizerOptions(base_options=base_options)
    recognizer = vision.GestureRecognizer.create_from_options(recognizer_options)

    mp_image = mp.Image.create_from_file(sample_image_path)
    result = recognizer.recognize(mp_image)

    if result.gestures and len(result.gestures) > 0:
        top_gesture = result.gestures[0][0]
        print(f"Predicted mudra: {top_gesture.category_name}")
        print(f"Confidence:      {top_gesture.score:.4f}")
    else:
        print("No gesture detected in the sample image.")
        print("This may happen if MediaPipe cannot detect a hand in the image.")

    recognizer.close()

    # Display the sample image
    img = cv2.imread(sample_image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.title(f"Sample: {sample_class}")
    plt.axis('off')
    plt.show()

Sample image: /content/drive/MyDrive/mudras/alapadma/alapadma.jpg
True label:   alapadma

Predicted mudra: alapadma
Confidence:      0.9664
